In [1]:
!sudo apt-get install -y zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 45 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (466 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently i

In [2]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
import subprocess
import time

# 서버를 백그라운드에서 실행
subprocess.Popen(["ollama", "serve"])
time.sleep(3) # 서버가 완전히 켜질 때까지 잠시 대기

In [4]:
!ollama pull llama3.1

In [6]:
!pip install -qU langchain-ollama

In [7]:
!pip install langchain-text-splitters langchain-postgres

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.8/212.8 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 93.0 MB/s eta 0:00:00


In [9]:
from langchain_ollama import ChatOllama

model=ChatOllama(model="llama3.1")


In [10]:
#모델에 문자열에 대한 list를 전달한다.
model.invoke("sky")

AIMessage(content="The sky! It's a vast and wondrous sight, covering our entire planet and shaping our experiences of weather, mood, and even inspiration. Here are some interesting aspects of the sky:\n\n1.  **Weather Patterns**: The sky plays a crucial role in determining our local weather. Clouds form when water vapor in the air condenses into visible liquid droplets. This can lead to various types of precipitation or keep skies clear and sunny.\n\n2.  **Stellar Visibility**: On a clear night, millions of stars are visible from Earth. The sky's darkness allows us to see these celestial bodies against the backdrop of space.\n\n3.  **Daylight Hours**: During different seasons, the tilt of Earth’s axis causes variations in daylight hours across the globe. This affects both the duration and quality of sunlight we receive each day.\n\n4.  **Seasonal Changes**: The sky's appearance changes throughout the year due to seasonal shifts. In some parts of the world, winter brings longer nights a

In [12]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

model=ChatOllama(model="llama3.1")
#HumanMessage라는 method로 사용자가 작성한 메시지 전달
prompt=[HumanMessage("프랑스의 수도는 어디인가요?")]

model.invoke(prompt)

AIMessage(content='파리입니다.', additional_kwargs={}, response_metadata={'model': 'llama3.1', 'created_at': '2026-03-28T01:24:55.108864041Z', 'done': True, 'done_reason': 'stop', 'total_duration': 414193702, 'load_duration': 194072587, 'prompt_eval_count': 19, 'prompt_eval_duration': 67626407, 'eval_count': 5, 'eval_duration': 139249948, 'logprobs': None, 'model_name': 'llama3.1', 'model_provider': 'ollama'}, id='lc_run--019d320b-4d24-74a2-9777-053542cd86ed-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 19, 'output_tokens': 5, 'total_tokens': 24})

In [25]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_ollama.chat_models import ChatOllama

model=ChatOllama(model="llama3.1")
#SystemMessage라는 method로 AI가 준수할 지침을 전달
system_msg=SystemMessage(
    "당신은 문장 끝에 느낌표(!)를 2개 이상 붙여 대답하는 친절한 어시스턴트입니다."
)

human_msg=HumanMessage("프랑스의 수도는 어디인가요?")

#모델에 지침과 사용자 입력에 대한 list를 전달한다.
model.invoke([system_msg,human_msg])

#출력은 다음과 같다.
#AIMessage(content='파리!! 프랑스의 수도가 파리야!!!')

AIMessage(content='파리!! 프랑스의 수도가 파리야!!!', additional_kwargs={}, response_metadata={'model': 'llama3.1', 'created_at': '2026-03-28T01:30:30.038883202Z', 'done': True, 'done_reason': 'stop', 'total_duration': 656561744, 'load_duration': 203588381, 'prompt_eval_count': 55, 'prompt_eval_duration': 21994312, 'eval_count': 12, 'eval_duration': 400505006, 'logprobs': None, 'model_name': 'llama3.1', 'model_provider': 'ollama'}, id='lc_run--019d3210-6885-7480-8e3d-675f2f4461fe-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 55, 'output_tokens': 12, 'total_tokens': 67})

In [31]:
#동적 prompt을 만든다.
from langchain_core.prompts import PromptTemplate

#template은 최종 prompt의 구조와 동적 입력이 삽입될 위치(Context: {context}와 같이)에 대한 정의로 구성된다.
template=PromptTemplate.from_template('''아래 작성한 Context를 기반으로 Question에 대답하세요. 제공된 정보로 대답할 수 없는 질문이라면 "모르겠어요."라고 답하세요.

Context: {context}

Question: {question}

Answer: ''')

template.invoke({
    'context': '''LLM은 NLP 분야의 최신 발전을 이끌고 있습니다.
    거대 언어 모델은 더 작은 모델보다 우수한 성능을 보이며, NLP 기능을 갖춘 애플리케이션을 개발하는 개발자들에게 매우 중요한 도구가 되었습니다.
    개발자들은 Huggin Face의 'transformers' 라이브러리를 활용하거나,
    'openai' 및 'cohere' 라이브러리를 통해 OpenAI와 Cohere의 서비스를 이용하여 LLM을 활용할 수 있습니다.
    ''',
    'question':'LLM은 어디서 제공하나요?'
})

StringPromptValue(text='아래 작성한 Context를 기반으로 Question에 대답하세요. 제공된 정보로 대답할 수 없는 질문이라면 "모르겠어요."라고 답하세요.\n\nContext: LLM은 NLP 분야의 최신 발전을 이끌고 있습니다.\n    거대 언어 모델은 더 작은 모델보다 우수한 성능을 보이며, NLP 기능을 갖춘 애플리케이션을 개발하는 개발자들에게 매우 중요한 도구가 되었습니다.\n    개발자들은 Huggin Face의 \'transformers\' 라이브러리를 활용하거나,\n    \'openai\' 및 \'cohere\' 라이브러리를 통해 OpenAI와 Cohere의 서비스를 이용하여 LLM을 활용할 수 있습니다.\n    \n\nQuestion: LLM은 어디서 제공하나요?\n\nAnswer: ')

In [32]:
#ollama의 LLM에 위에서 만든 동적 프롬프트를 입력하겠다.
prompt= template.invoke({
    'context': '''LLM은 NLP 분야의 최신 발전을 이끌고 있습니다.
    거대 언어 모델은 더 작은 모델보다 우수한 성능을 보이며, NLP 기능을 갖춘 애플리케이션을 개발하는 개발자들에게 매우 중요한 도구가 되었습니다.
    개발자들은 Huggin Face의 'transformers' 라이브러리를 활용하거나,
    'openai' 및 'cohere' 라이브러리를 통해 OpenAI와 Cohere의 서비스를 이용하여 LLM을 활용할 수 있습니다.
    ''',
    'question':'LLM은 어디서 제공하나요?'
})

completion=model.invoke(prompt)

print(completion)

content="Huggin Face의 'transformers' 라이브러리를 활용하거나, 'openai' 및 'cohere' 라이브러리를 통해 OpenAI와 Cohere의 서비스를 이용하여 LLM을 활용할 수 있습니다." additional_kwargs={} response_metadata={'model': 'llama3.1', 'created_at': '2026-03-28T01:42:27.430363157Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2000119092, 'load_duration': 200911231, 'prompt_eval_count': 179, 'prompt_eval_duration': 259983151, 'eval_count': 50, 'eval_duration': 1442253245, 'logprobs': None, 'model_name': 'llama3.1', 'model_provider': 'ollama'} id='lc_run--019d321b-5594-70b0-8ad4-0c4bc642420d-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 179, 'output_tokens': 50, 'total_tokens': 229}


In [36]:
#ChatPromptTemplate라는 method를 활용하여 채팅 메시지의 역할에 따른 동적 프롬프트를 제공할 수 있다.
from langchain_core.prompts import ChatPromptTemplate

template=ChatPromptTemplate.from_messages([
    ('system','''아래 작성한 Context를 기반으로 Question에 대답하세요.
    제공된 정보로 대답할 수 없는 질문이라면 "모르겠어요."라고 답하세요.'''),  #LLM이 지켜야할 지침
    ('human','Context: {context}'),     #사용자가 채팅으로 제공할 context
    ('human','Question: {question}')    #사용자가 채팅으로 제공할 question
])

template.invoke({
    'context': '''LLM은 NLP 분야의 최신 발전을 이끌고 있습니다.
    거대 언어 모델은 더 작은 모델보다 우수한 성능을 보이며, NLP 기능을 갖춘 애플리케이션을 개발하는 개발자들에게 매우 중요한 도구가 되었습니다.
    개발자들은 Huggin Face의 'transformers' 라이브러리를 활용하거나,
    'openai' 및 'cohere' 라이브러리를 통해 OpenAI와 Cohere의 서비스를 이용하여 LLM을 활용할 수 있습니다.
    ''',
    'question':'LLM은 어디서 제공하나요?'
})

"""
HumanMessage(content="Context: LLM은 NLP 분야의 최신 발전을 이끌고 있습니다.\n
거대 언어 모델은 더 작은 모델보다 우수한 성능을 보이며, NLP 기능을 갖춘 애플리케이션을 개발하는 개발자들에게 매우 중요한 도구가 되었습니다.\n
개발자들은 Huggin Face의 'transformers' 라이브러리를 활용하거나,\n
'openai' 및 'cohere' 라이브러리를 통해 OpenAI와 Cohere의 서비스를 이용하여 LLM을 활용할 수 있습니다.\n")


HumanMessage(content='Question: LLM은 어디서 제공하나요?')

prompt는 지시문을 담은 하나의 SystemMessage와 동적 변수인 context와 question을 담고 있는 두 개의 HumanMessage를 담고있는 것을 알 수 있다.
"""

ChatPromptValue(messages=[SystemMessage(content='아래 작성한 Context를 기반으로 Question에 대답하세요.\n    제공된 정보로 대답할 수 없는 질문이라면 "모르겠어요."라고 답하세요.', additional_kwargs={}, response_metadata={}), HumanMessage(content="Context: LLM은 NLP 분야의 최신 발전을 이끌고 있습니다.\n    거대 언어 모델은 더 작은 모델보다 우수한 성능을 보이며, NLP 기능을 갖춘 애플리케이션을 개발하는 개발자들에게 매우 중요한 도구가 되었습니다.\n    개발자들은 Huggin Face의 'transformers' 라이브러리를 활용하거나,\n    'openai' 및 'cohere' 라이브러리를 통해 OpenAI와 Cohere의 서비스를 이용하여 LLM을 활용할 수 있습니다.\n    ", additional_kwargs={}, response_metadata={}), HumanMessage(content='Question: LLM은 어디서 제공하나요?', additional_kwargs={}, response_metadata={})])

In [34]:
#위에서 만든 동적 프롬프트를 적용한 LLM호출
prompt= template.invoke({
    'context': '''LLM은 NLP 분야의 최신 발전을 이끌고 있습니다.
    거대 언어 모델은 더 작은 모델보다 우수한 성능을 보이며, NLP 기능을 갖춘 애플리케이션을 개발하는 개발자들에게 매우 중요한 도구가 되었습니다.
    개발자들은 Huggin Face의 'transformers' 라이브러리를 활용하거나,
    'openai' 및 'cohere' 라이브러리를 통해 OpenAI와 Cohere의 서비스를 이용하여 LLM을 활용할 수 있습니다.
    ''',
    'question':'LLM은 어디서 제공하나요?'
})

model.invoke(prompt)

AIMessage(content='Huggin Face, openai, cohere의 라이브러리나 서비스를 통하여 LLM을 사용할 수 있습니다.', additional_kwargs={}, response_metadata={'model': 'llama3.1', 'created_at': '2026-03-28T01:47:48.635994636Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5201602175, 'load_duration': 3969308934, 'prompt_eval_count': 182, 'prompt_eval_duration': 264745760, 'eval_count': 28, 'eval_duration': 910211156, 'logprobs': None, 'model_name': 'llama3.1', 'model_provider': 'ollama'}, id='lc_run--019d3220-2fc2-7bc0-8841-5261bd4474f0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 182, 'output_tokens': 28, 'total_tokens': 210})

In [39]:
#LLM의 출력을 JSON형식으로 설정
from pydantic import BaseModel

#LLM이 출력할 JSON의 스키마를 정의
class AnswerWithJustification(BaseModel): #class BaseModel을 상속받는다.
  answer:str    #사용자의 질문에 대한 답변

  justification: str    #그 답변에 대한 근거

llm=ChatOllama(model='llama3.1',temperature=0)
structured_llm=llm.with_structured_output(AnswerWithJustification)

#LLM은 위에서 정의한 schema를 기반으로 아래 질문에 대한 답을 생성한다.
result=structured_llm.invoke('''1kg의 벽돌과 1kg의 깃털 중 어느쪽이 더 무겁나요?''')

print(result.model_dump_json())


{"answer":"깃털","justification":"깃털은 공기 중에 떠올라서 실제로 무게가 적습니다."}


In [42]:
#langchain의 CSV출력 파서(output parser는 LLM 응답을 구조화하는 class이다.)

from langchain_core.output_parsers import CommaSeparatedListOutputParser

#,를 기준으로 나눠 list로 구조화한다.
parser=CommaSeparatedListOutputParser()
items=parser.invoke("apple,banna,cherry")
print(items)

['apple', 'banna', 'cherry']


#langchain의 공통 interface 예>

In [43]:
#invoke
from langchain_ollama.chat_models import ChatOllama

model=ChatOllama(model="llama3.1",temperature=0)

completion=model.invoke("반가워요!")

print(completion)

content='반가워요! 안녕하세요. 어떻게 지내세요?' additional_kwargs={} response_metadata={'model': 'llama3.1', 'created_at': '2026-03-28T02:04:00.629563921Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4664658548, 'load_duration': 4160216403, 'prompt_eval_count': 15, 'prompt_eval_duration': 68011569, 'eval_count': 14, 'eval_duration': 395510928, 'logprobs': None, 'model_name': 'llama3.1', 'model_provider': 'ollama'} id='lc_run--019d322f-06ba-7103-ae03-7eba4003abd8-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 15, 'output_tokens': 14, 'total_tokens': 29}


In [44]:
#batch
completions=model.batch(['반가워요!','잘 있어요!']) #입력은 list 형태이다.
print(completions)

#[AIMessage(content='반가워요! 안녕하세요. 어떻게 지내세요?'), AIMessage(content='잘 지내고 계시나요?')]

[AIMessage(content='반가워요! 안녕하세요. 어떻게 지내세요?', additional_kwargs={}, response_metadata={'model': 'llama3.1', 'created_at': '2026-03-28T02:05:19.373787892Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1622441302, 'load_duration': 1137629550, 'prompt_eval_count': 15, 'prompt_eval_duration': 34049120, 'eval_count': 14, 'eval_duration': 404629743, 'logprobs': None, 'model_name': 'llama3.1', 'model_provider': 'ollama'}, id='lc_run--019d3230-462b-7750-80fa-c9ee8f301d3c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 14, 'total_tokens': 29}), AIMessage(content='잘 지내고 계시나요?', additional_kwargs={}, response_metadata={'model': 'llama3.1', 'created_at': '2026-03-28T02:05:19.710031976Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1959768969, 'load_duration': 1168181209, 'prompt_eval_count': 15, 'prompt_eval_duration': 48523496, 'eval_count': 10, 'eval_duration': 257320648, 'logprobs': None, 'model_name': 'llama3.1', 'model_provid

In [45]:
#stream
for token in model.stream("잘 있어요!"):
  print(token)

content='잘' additional_kwargs={} response_metadata={} id='lc_run--019d3231-9a10-7800-9568-9d4ca7945fae' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content=' 지' additional_kwargs={} response_metadata={} id='lc_run--019d3231-9a10-7800-9568-9d4ca7945fae' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content='내' additional_kwargs={} response_metadata={} id='lc_run--019d3231-9a10-7800-9568-9d4ca7945fae' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content='고' additional_kwargs={} response_metadata={} id='lc_run--019d3231-9a10-7800-9568-9d4ca7945fae' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content=' 계' additional_kwargs={} response_metadata={} id='lc_run--019d3231-9a10-7800-9568-9d4ca7945fae' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content='시' additional_kwargs={} response_metadata={} id='lc_run--019d3231-9a10-7800-9568-9d4ca7945fae' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content='나요' additional_kwargs={} resp

#langchain 요소의 구성

In [48]:
#명령형 구성
#prompt와 chating model을 사용해 구성한 chatbot이다.
from langchain_ollama.chat_models import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import chain

#구성 요소
template=ChatPromptTemplate.from_messages(
    [
        ('system','당신은 친절한 어시스턴트입니다.'),
        ('human','{question}'),
    ]
)

model=ChatOllama(model="llama3.1")

#함수로 결합한다.
#데코레이터 @chain을 추가해 작성한 함수에 Runnable interface를 추가한다.
@chain
def chatbot(values):
  prompt=template.invoke(values)
  return model.invoke(prompt)

#사용한다.
response=chatbot.invoke({'question':'거대 언어 모델은 어디서 제공하나요?'})
print(response)

content='다음은 몇 가지 주요 플랫폼과 도구를 나열합니다.\n\n1. **Hugging Face Transformers**: 오픈 소스 라이브러리, 100을 초과하는 언어 모델이 있습니다.\n2. **Google Cloud AI Platform**: Google 클라우드의 AI 플랫폼입니다.\n3. **AWS SageMaker**: AWS에서 제공하는 머신 러닝 서비스입니다.\n4. **Microsoft Azure Machine Learning**: Microsoft Azure의 머신 러닝 서비스입니다.\n5. **IBM Watson Studio**: IBM Watson Studio는 사용자에게 AI 및 데이터 과학을 위한 통합 개발 환경(IDE)을 제공합니다.\n\n이들 플랫폼 및 도구를 이용하여 거대 언어 모델을 개발, 트레이닝 할 수 있습니다.' additional_kwargs={} response_metadata={'model': 'llama3.1', 'created_at': '2026-03-28T02:12:58.718972046Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6847898382, 'load_duration': 1314910545, 'prompt_eval_count': 39, 'prompt_eval_duration': 39013535, 'eval_count': 161, 'eval_duration': 5013390278, 'logprobs': None, 'model_name': 'llama3.1', 'model_provider': 'ollama'} id='lc_run--019d3237-340f-7e82-9863-54a978fea8b2-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 39, 'output_tokens': 161, 'total_tokens': 200}


In [49]:
#명령형 구성을 사용한 streaming 호출 예시
@chain
def chatbot(values):
  prompt=template.invoke(values)
  for token in model.stream(prompt):
    yield token

for part in chatbot.stream({
    'question':'거대 언어 모델은 어디서 제공하나요?'
}):
  print(part)

content='거' additional_kwargs={} response_metadata={} id='lc_run--019d323a-b5f3-7911-a71a-f4024bc3f216' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content='대한' additional_kwargs={} response_metadata={} id='lc_run--019d323a-b5f3-7911-a71a-f4024bc3f216' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content=' 언어' additional_kwargs={} response_metadata={} id='lc_run--019d323a-b5f3-7911-a71a-f4024bc3f216' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content=' 모델' additional_kwargs={} response_metadata={} id='lc_run--019d323a-b5f3-7911-a71a-f4024bc3f216' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content='을' additional_kwargs={} response_metadata={} id='lc_run--019d323a-b5f3-7911-a71a-f4024bc3f216' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content=' 다양한' additional_kwargs={} response_metadata={} id='lc_run--019d323a-b5f3-7911-a71a-f4024bc3f216' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content=' 곳' additional_kwargs={

In [50]:
#명령형 구성을 사용한 비동기 실행
@chain
async def chatbot(values):
  prompt=await template.ainvoke(values)
  return await model.ainvoke(prompt)

await chatbot.ainvoke({'question':'거대 언어 모델은 어디서 제공하나요?'})

AIMessage(content='거대한 언어 모델(또는 대규모 언어 모델)은 다양한 곳에서 제공됩니다.\n\n1. **Hugging Face Transformers**: Hugging Face는 많은 인기 있는 언어 모델, 예를 들어 BERT, RoBERTa 및 XLNet 등이 포함된 Transforme가있는 개방형 라이브러리를 구축하고 있습니다.\n2. **Transformers(Transformer Repository)**: Transformers는 GitHub에 호스팅되며, OpenNMT, BART 및 T5와 같은 언어 모델을 포함합니다.\n3. **PyTorch**: PyTorch는 다양한 언어 모델이 포함된 개방형 머신 러닝 프레임워크입니다. PyTorch에 호스팅되는 모델은 많은 경우 Hugging Face Transformers와 중복되지만 일부 모델은 독점적입니다.\n4. **TensorFlow**: TensorFlow도 여러 언어 모델을 호스트하고 있습니다.텐서플로의 개방형 모델을 사용하면 TF 모델을 훈련 및 평가할 수 있습니다.\n5. **Microsoft Azure Machine Learning**: Microsoft는 몇몇 대규모 언어 모델, 예를 들어 Turing-NLG와 같은)을 호스팅합니다.\n6. **Google Cloud AI Platform**: Google은 BERT와 같은 일부 주요 모델을 호스트하고 있습니다.\n\n그냥 적어도 참고하세요!\n\n언어가 모델의 제공 위치는 지속적으로 변할 수 있으니 최신 정보를 얻기 위해 공식문서 및 공식 소스 코드에 방문해 보세요!', additional_kwargs={}, response_metadata={'model': 'llama3.1', 'created_at': '2026-03-28T02:18:48.065308302Z', 'done': True, 'done_reason': 'stop', 'total_duration': 12717682728, 'load_duration': 2396

In [51]:
#선언형 구성
from langchain_ollama.chat_models import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

#구성 요소
template=ChatPromptTemplate.from_messages(
    [
        ('system','당신은 친절한 어시스턴트입니다.'),
        ('human','{question}'),
    ]
)

model=ChatOllama(model="llama3.1")

#연산자 |로 결합한다.
chatbot=template|model

response=chatbot.invoke({'question':'거대 언어 모델은 어디서 제공하나요?'})

In [53]:
response

AIMessage(content='거대한 언어 모델은 여러 곳에서 제공됩니다.\n\n1. **Hugging Face Transformers**: Hugging Face는 다양한 오픈 소스 언어 모델을 호스팅하는 유명한 플랫폼입니다. 그들 중 가장 유명한 것들은 BERT, RoBERTa, XLNet 및 DistilBERT입니다.\n2. **GitHub**: GitHub은 많은 개발자와 연구원들이 자신의 언어 모델을 공유하는 곳입니다. GitHub에서 다양한 종류의 언어 모델이 있습니다.\n3. **AllenNLP**: AllenNLP는 연구 단체인 Allen AI가 제공하는 오픈 소스 자연어 처리 라이브러리입니다. AllenNLP에는 여러 언어 모델이 포함되어 있습니다.\n4. **NLTK**: NLTK(Natural Language Toolkit)는 파이썬을 위한 오픈 소스 자연어 처리 도구 모음입니다. NLTK는 여러 언어 모델을 제공합니다.\n\n이러한 거대 언어 모델은 AI와 NLP의 연구를 위해 매우 유용합니다. 다만, 사용하기 전에는 각 모델의 특성과 제약조건에 대해 잘 이해해야 합니다.', additional_kwargs={}, response_metadata={'model': 'llama3.1', 'created_at': '2026-03-28T02:22:50.548692945Z', 'done': True, 'done_reason': 'stop', 'total_duration': 7818044861, 'load_duration': 196342699, 'prompt_eval_count': 39, 'prompt_eval_duration': 35613623, 'eval_count': 242, 'eval_duration': 6983226103, 'logprobs': None, 'model_name': 'llama3.1', 'model_provider': 'ollama'}, id='lc_run--019d3240-3829-7433-8a95-52e93fb81cc

In [54]:
#선언형 구성을 사용한 스트리밍 호출 예시
chatbot=template|model

for part in chatbot.stream({
    "question":"거대 언어 모델은 어디서 제공하나요?"
}):
  print(part)

content='거' additional_kwargs={} response_metadata={} id='lc_run--019d3241-e06b-7a92-918c-40851017aa8c' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content='대한' additional_kwargs={} response_metadata={} id='lc_run--019d3241-e06b-7a92-918c-40851017aa8c' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content=' 언어' additional_kwargs={} response_metadata={} id='lc_run--019d3241-e06b-7a92-918c-40851017aa8c' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content=' 모델' additional_kwargs={} response_metadata={} id='lc_run--019d3241-e06b-7a92-918c-40851017aa8c' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content='은' additional_kwargs={} response_metadata={} id='lc_run--019d3241-e06b-7a92-918c-40851017aa8c' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content=' 다양한' additional_kwargs={} response_metadata={} id='lc_run--019d3241-e06b-7a92-918c-40851017aa8c' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content=' 플' additional_kwargs={

In [55]:
#선언형 구성을 사용한 비동기 실행
chatbot=template|model

await chatbot.ainvoke({
    'question':'거대 언어 모델은 어디서 제공하나요?'
})

AIMessage(content='현재 거대한 언어 모델은 여러 곳에서 제공되고 있습니다. 이 중 일부는 open-source이거나 무료이며, 다른 곳에서는 유료로 제공되고 있습니다.\n\n대표적인 거대 언어 모델의 목록은 다음과 같습니다:\n\n1. **Hugging Face Transformers**: Hugging Face는 다양한 언어 모델을 제공하고 있는 것으로-known합니다. 그 중 가장 유명한 모델은 BERT, RoBERTa 등이 있습니다.\n2. **Google Cloud AI Platform**: Google Cloud AI Platform에서는 여러 가지 거대 언어 모델을 제공하고 있습니다. 예를 들어, T5, Longformer 등이 있습니다.\n3. **Microsoft Azure Machine Learning**: Microsoft Azure Machine Learning에서도 다양한 거대 언어 모델을 제공하고 있습니다. 예를 들어, DALL-E, CLIP 등이 있습니다.\n4. **IBM Watson Studio**: IBM Watson Studio에서는 여러 가지 거대 언어 모델을 제공하고 있습니다. 예를 들어, BERT, RoBERTa 등이 있습니다.\n5. **Open AI Gym**: Open AI Gym은 다양한 거대 언어 모델을 제공하고 있습니다. 예를 들어, Transformer-XL, XLNet 등이 있습니다.\n\n이 외에도 여러 가지 다른 곳에서 거대 언어 모델을 제공하고 있는 것을 찾을 수 있을 것입니다.', additional_kwargs={}, response_metadata={'model': 'llama3.1', 'created_at': '2026-03-28T02:25:29.257494671Z', 'done': True, 'done_reason': 'stop', 'total_duration': 8549411700, 'load_duration': 563602992, 'prompt_eval_count': 39, 